# 先上线：最朴素的部署方案

## 明确目标，清点行囊

现在 zero-to-tech 项目由三部分组成：

- **静态前端**：基于 React 和 Next.js 开发，可以构建为静态前端文件。
- **FastAPI 后端**：基于 Python 和 FastAPI 开发，通过 Uvicorn 提供服务。
- **SQLite**：对应硬盘上的一个 `.db` 文件。

上次部署到服务器的还只是静态前端。现在项目已经增加了后端 API、数据库、历史记录和会话功能，但这些都只在本地运行。

因此，这次全栈部署需要同时处理前端和后端：

- 前端推送最新代码并重新构建。
- 后端从零开始准备环境并运行。
- SQLite 跟随后端，在代码启动时自动创建。

模块 7 将依次完成：

1. **7.1**：用最朴素的办法把整套项目跑通并上线。
2. **7.2**：改用更易维护的部署方式，并通过反向代理实现前后端同源。
3. **7.3**：域名与 HTTPS。
4. **7.4**：通过日志和统计数据观察用户行为。
5. **7.5**：调优、安全和一键发布。

## 如何部署

部署说到底就是：

> 把在自己电脑上能运行的项目，放到另一台机器，尤其是服务器上运行。

SQLite 的安装、创建和使用都依赖 Python，可以把它看作后端的一部分。因此，本节主要部署前端和后端。

## 朴素的部署方案

项目在本地运行时：

- 前端通过 `npm run dev` 监听 3000 端口。
- 后端通过 `fastapi dev` 监听 8000 端口。
- 浏览器访问前端，前端 JavaScript 再通过 `fetch` 请求后端。

服务器也是一台电脑，所以先采用最直接的方案：

~~~text
浏览器
  ├── http://服务器IP          → Nginx:80 → 静态前端
  └── http://服务器IP:8000     → Uvicorn  → FastAPI 后端
~~~

前端不再用开发服务器监听 3000 端口，而是构建成静态文件，继续由 Nginx 通过 80 端口提供。

后端暂时直接通过 Uvicorn 在 8000 端口提供服务。

## 我们已经知道的

### 1. 前端部署

前端代码推送到服务器后，安装依赖、构建静态文件，再交给 Nginx 提供。模块 3 和模块 4 已经完成过这一流程。

需要注意：5.5 新增的 `.env.local` 不被 Git 追踪，服务器上不会自动出现，需要单独处理。

### 2. Python 环境

Ubuntu 通常自带 Python。后端运行前需要确认 Python 已安装且版本足够。

### 3. 虚拟环境与依赖

项目使用 `.venv` 管理 Python 依赖，并通过 `requirements.txt` 记录依赖。服务器上可以重建虚拟环境，再执行：

~~~bash
pip install -r requirements.txt
~~~

### 4. 代码和数据库

`main.py`、`storage.py` 等代码通过 Git 和 GitHub 同步。`history.db` 不进 Git，但后端启动时会自动创建。

### 5. 启动后端

本地一直使用 `fastapi dev`，背后仍然由 Uvicorn 监听 8000 端口并提供 API。

## 服务器部署会遇到的挑战

### 挑战 1：缺少前端配置文件

`.env.local` 不进 Git，服务器拉取代码后没有前端配置。而且 `.local` 表示本机私有，线上生产构建应使用另一种配置文件。

### 挑战 2：后端配置不匹配

后端 CORS 目前写死了本地地址：

~~~python
allow_origins=["http://localhost:3000"]
~~~

线上用户不会从 `localhost:3000` 访问。这种随环境变化的值不应该写死在代码里，也应该放入配置文件。

### 挑战 3：`fastapi dev` 只适合本地开发

`fastapi dev` 默认只监听本机地址，外部用户无法通过互联网连接。线上应使用 `fastapi run`。

### 挑战 4：终端关闭后端就停止

后端目前是终端中的前台进程。SSH 窗口一关，服务就停止。

### 挑战 5：8000 端口没有放行

云平台安全组或防火墙通常没有开放 8000 端口，需要手动放行。

五条挑战归拢后是四件事：

1. 在本地处理前后端配置文件。
2. 把 `fastapi dev` 换成 `fastapi run`。
3. 在云平台开放 8000 端口。
4. 最后让后端在后台持续运行。

## 配置文件

前端和后端的问题本质相同：有些值会随着运行环境变化，不应该写死在代码中。

### 什么是配置？

同一份代码在本机和服务器上运行时，前后端地址会不同。适合放入配置的内容还包括：

- 前后端地址。
- 数据库地址、连接信息和密码。
- 第三方服务密钥，例如大模型 API Key。
- 服务监听的端口。
- 日志位置和调试开关。

配置文件通常使用“名称=值”的形式：

~~~dotenv
名称a=xxx
名称b=123
~~~

代码回答“怎么做”，配置回答“这一次对着谁做”。同一份代码配上不同配置，就能运行在不同环境。

## 项目中已有的两处地址配置

前端根目录中的 `.env.local`：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=http://localhost:8000
~~~

前端通过 `NEXT_PUBLIC_API_BASE_URL` 找到后端地址。

后端 `main.py` 中则写死了：

~~~python
allow_origins=["http://localhost:3000"]
~~~

它表示允许哪个前端源跨源访问后端。这次要把它也抽到配置文件中。

## 配置文件的命名和读取

配置文件通常叫 `.env`，但具体名称由框架约定。

Next.js 会识别：

- `.env.local`：本机私有配置。
- `.env.development.local`。
- `.env.development`。
- `.env.production`：生产构建使用。
- `.env`：各环境通用，优先级较低。

`npm run dev` 时，常见查找优先级为：

~~~text
.env.development.local
→ .env.local
→ .env.development
→ .env
~~~

`npm run build` 时，中间的 `development` 会换成 `production`。

Python 后端使用 `python-dotenv` 读取 `.env`。前端配置放在项目根目录，后端配置放在 `backend/` 目录。以点开头的文件在 Linux 中是隐藏文件，这也是配置文件常见的命名习惯。

## 配置文件不能进入 Git

配置文件通常与某台机器、某个环境有关，不需要随着代码同步。

更重要的是，配置文件可能包含密码和密钥，绝对不能推送到远程仓库，尤其不能推到 GitHub 公开仓库。

因此要在 `.gitignore` 中明确忽略真实配置文件。

## README 和 `.env.example`

真实配置不进 Git 会带来一个问题：别人拿到代码后，不知道要创建哪些配置、文件放在哪里、键名是什么。

项目根目录中的 `README.md` 应说明：

- 项目解决什么问题。
- 项目结构和技术栈。
- 如何安装和运行。
- 需要创建哪些配置文件。

还应提供一份示例配置，例如 `.env.example`。它与真实 `.env` 的键名和格式相同，但值是示例，不包含秘密，因此可以进入 Git。

使用者可以复制示例，再修改实际值：

~~~bash
cp .env.example .env
~~~

## 动手处理后端配置

先在 `backend/` 中创建 `.env`：

~~~dotenv
# backend/.env
ALLOWED_ORIGINS=http://localhost:3000
~~~

确认 `.gitignore` 中有：

~~~gitignore
.env*
~~~

然后执行：

~~~bash
git status
~~~

`backend/.env` 不应该出现在待提交列表中。配置不进 Git 不能只靠记忆，要亲自通过 `git status` 验证。

## 让 Python 读取配置

修改 `backend/main.py`。顶部加入：

~~~python
import os
from dotenv import load_dotenv
~~~

读取配置并拆分允许的源：

~~~python
load_dotenv()
ALLOWED_ORIGINS = os.getenv("ALLOWED_ORIGINS").split(",")
~~~

把 CORS 中写死的地址替换掉：

~~~python
app.add_middleware(
    CORSMiddleware,
    allow_origins=ALLOWED_ORIGINS,
    allow_credentials=True,
    allow_methods=["GET", "POST"],
)
~~~

`python-dotenv` 只负责把 `.env` 中的内容读入环境变量。`os.getenv()` 再从环境变量中读取指定值。

## 安装 `python-dotenv` 并记账

`dotenv` 不是 Python 标准库，需要安装，并重新生成依赖清单：

~~~bash
cd backend
source .venv/bin/activate
pip install python-dotenv
pip freeze > requirements.txt
~~~

本地启动前后端进行验证：

~~~bash
# 后端
fastapi dev

# 前端，另开终端
npm run dev
~~~

文字实验室仍然能正常使用，说明后端已经成功从配置中读取 CORS 地址。

如果要进一步确认，可以暂时把 `ALLOWED_ORIGINS` 改成其他地址并重启后端。分析请求会被浏览器拦截；再把配置改回来即可。

## 创建前后端配置示例

在项目根目录创建 `.env.example`：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=http(s)://[ip]:[port]
~~~

在 `backend/` 中创建 `.env.example`：

~~~dotenv
ALLOWED_ORIGINS=http(s)://[ip]:[port]
~~~

键名必须完整，值应提供可用示例或清晰指引。这样别人复制文件后只需修改值，不用猜键名。

## 让示例配置进入 Git

`.gitignore` 中的 `.env*` 也会匹配 `.env.example`，因此需要在它下面增加一个例外：

~~~gitignore
.env*
!.env.example
~~~

这表示忽略真实配置，但允许所有目录中的 `.env.example` 被 Git 追踪。

完成后提交本地修改：

~~~bash
git add -A
git commit -m "CORS 名单改为从配置读取;补 .env.example"
git push
~~~

本次提交包括：

- 读取配置的 `main.py`。
- 增加 `python-dotenv` 的 `requirements.txt`。
- 前后端两份 `.env.example`。
- 更新后的 `.gitignore`。

## 服务器部署：拉取代码

SSH 登录服务器，然后拉取最新代码：

~~~bash
ssh 用户名@你的服务器IP

cd ~/zero-to-tech
git pull
~~~

检查后端目录：

~~~bash
ls backend/
~~~

应当看到：

~~~text
main.py
storage.py
requirements.txt
.env.example
~~~

这里应该没有 `.venv/`、`history.db` 和 `.env`。代码通过 Git 同步，依赖、运行时数据和真实配置不随代码同行。

## 准备 Python 环境

先检查服务器上的 Python：

~~~bash
python3 --version
which python3
~~~

两条命令回答不同问题：

- `python3 --version`：是否安装，版本是否满足依赖要求。
- `which python3`：当前使用的是哪一个 Python。

服务器可能同时存在多个 Python，`python` 和 `python3` 也可能指向不同位置。

Python 3.10 或更高版本通常可以满足本项目。如果版本过低或未安装，需要先安装合适的 Python。

Node.js 在前端部署时已经安装过，不放心可以检查：

~~~bash
node -v
~~~

## 在服务器创建虚拟环境并安装依赖

进入后端目录：

~~~bash
cd ~/zero-to-tech/backend
python3 -m venv --prompt=zero-to-tech .venv
source .venv/bin/activate
pip install -r requirements.txt
~~~

激活后，终端提示符会出现 `(zero-to-tech)`。

如果 Ubuntu 提示缺少 venv 组件，按照提示安装，例如：

~~~bash
sudo apt install python3-venv
~~~

某些版本会提示更具体的包名，例如 `python3.12-venv`。安装完成后重新创建虚拟环境。

## 在服务器写前端配置

进入项目根目录，从示例复制生产配置：

~~~bash
cd ~/zero-to-tech
cp .env.example .env.production
vim .env.production
~~~

写入：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=http://服务器IP:8000
~~~

前端生产构建时会读取 `.env.production`，生成的静态 JavaScript 将通过这个地址请求后端。

## 在服务器写后端配置

进入后端目录：

~~~bash
cd ~/zero-to-tech/backend
cp .env.example .env
vim .env
~~~

写入：

~~~dotenv
ALLOWED_ORIGINS=http://服务器IP
~~~

这里不要写 `:3000`，也不需要写 `:80`。

| 配置 | 本地 | 线上 |
| --- | --- | --- |
| `NEXT_PUBLIC_API_BASE_URL` | `http://localhost:8000` | `http://服务器IP:8000` |
| `ALLOWED_ORIGINS` | `http://localhost:3000` | `http://服务器IP` |

`ALLOWED_ORIGINS` 登记的是前端页面所在的源。本地前端运行在 3000 端口；线上前端由 Nginx 在 80 端口提供，而 80 是 HTTP 默认端口，浏览器发出的 Origin 中不会写出它。

配置值要跟着服务器上的事实变化，不能机械地照抄本地地址。

## 跑起来：构建前端

进入项目根目录：

~~~bash
cd ~/zero-to-tech
npm install
npm run build
~~~

Next.js 读取 `.env.production`，构建生成静态前端文件。Nginx 继续通过 80 端口提供这些静态资源。

## 跑起来：启动后端

进入后端目录并启动生产服务：

~~~bash
cd ~/zero-to-tech/backend
source .venv/bin/activate
fastapi run
~~~

启动日志会显示类似：

~~~text
FastAPI Starting production server
Server started at http://0.0.0.0:8000
INFO: Uvicorn running on http://0.0.0.0:8000
~~~

背后仍然是 Uvicorn，但 `run` 与 `dev` 的用途不同：

| `fastapi dev` | `fastapi run` |
| --- | --- |
| 修改代码后自动重启 | 不自动重启 |
| 默认监听 `127.0.0.1` | 默认监听 `0.0.0.0` |
| 适合本机开发 | 适合服务器运行 |

- `127.0.0.1` 只接受服务器内部连接。
- `0.0.0.0` 监听所有网卡，外部用户也能连接。

## 先在服务器本地验证

在后端运行期间，再打开一个终端并重新 SSH 登录。

先测试后端接口：

~~~bash
curl localhost:8000/api/profile
~~~

能看到 JSON，说明后端已经启动，并且正在监听 8000 端口。

再检查数据库：

~~~bash
ls ~/zero-to-tech/backend/
~~~

如果出现 `history.db`，说明应用启动时已成功初始化 SQLite 数据库。

公网访问之前，应该先确认服务器本机访问正常。这样公网访问失败时，排查范围可以集中在端口和网络，而不是应用代码。

## 开通 8000 端口

前端通过 80 端口访问，后端暂时通过 8000 端口访问。80 端口此前已经开放，但云平台安全组或防火墙通常不会默认开放 8000。

在云平台控制台中添加入站规则，放行 TCP 8000 端口。操作方法与此前开放 80 端口相同。

开放后，在本地电脑访问：

~~~text
http://服务器IP:8000/api/profile
~~~

如果浏览器得到 JSON，说明 8000 端口已经能从公网访问。

## 完整线上测试

打开：

~~~text
http://服务器IP
~~~

依次验证：

1. 打开文字实验室，分析一句文字。它测试前端能否请求线上后端。
2. 打开历史记录，确认刚才的内容存在。它测试 SQLite 是否正常工作。
3. 换一个浏览器或打开无痕窗口，分析另一句话，再查看历史。
4. 两个浏览器应该只能看到各自的记录。它测试 6.6 完成的会话机制。

如果全部正常，zero-to-tech 已经可以通过电脑或手机浏览器从互联网访问。

## 服务端的后台运行

此时后端仍然是前台进程，寄生在当前 SSH 会话中。关闭 SSH 窗口后：

- Nginx 提供的静态前端仍然存在。
- Uvicorn 后端停止，分析按钮会请求失败。

我们希望后端不依赖 SSH 会话。重新登录服务器后执行：

~~~bash
cd ~/zero-to-tech/backend
nohup .venv/bin/fastapi run > backend.log 2>&1 &
~~~

这条命令有四个关键部分：

- 末尾的 `&`：把命令放到后台运行，终端不用一直等待。
- 开头的 `nohup`：no hang up，让进程不因 SSH 断开而被终止。只有 `&` 还不够。
- `> backend.log`：把正常输出重定向到日志文件。
- `2>&1`：让错误输出 2 也跟随正常输出 1，写入同一个日志文件。

执行后终端会显示一个数字，它是进程号 PID。

这里直接使用 `.venv/bin/fastapi` 的完整路径，不依赖当前终端是否激活虚拟环境。关闭 SSH 后再次访问网站，后端应该仍然工作。

## 查看日志和停止服务

这套朴素方案需要手动管理后端：

~~~bash
tail -f backend.log
~~~

实时查看日志。

~~~bash
ps aux | grep fastapi
~~~

查找 FastAPI 进程及其 PID。

~~~bash
kill 进程号
~~~

停止对应进程。

这些命令也将写入项目 README，方便以后部署和维护。

## 回顾并写入 `README.md`

部署完成并验证后，再把走通的步骤写进 README。

不是先凭想象写部署文档，而是实际完成后记录经过验证的路径。一份错误或过时的 README 会把使用者带进坑里。

README 至少要说明：

- 项目用途和技术栈。
- 本地如何启动前后端。
- 如何部署到服务器。
- 前后端配置文件的键名和示例。
- 如何启动、查看日志和停止后端。
- 需要放行哪些端口。
- 如何验证分析、历史记录和会话。

## README：项目与本地运行

~~~~markdown
# 文字实验室

一个中文文本分析小工具：输入一段话，给出情感倾向评分和全文拼音，
并保存每次分析结果，每个访客只看到自己的历史。

## 技术栈

- 前端：Next.js（静态导出）+ React
- 后端：FastAPI + Uvicorn
- 分析：SnowNLP、pypinyin
- 存储：SQLite
- 线上：Nginx

## 本地运行

需要 Node.js 18+、Python 3.10+。

### 后端

~~~bash
cd backend
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env
# 按配置说明修改 .env
fastapi dev
~~~

后端地址：`http://localhost:8000`

### 前端

另开一个终端：

~~~bash
npm install
cp .env.example .env.local
# 按配置说明修改 .env.local
npm run dev
~~~

前端地址：`http://localhost:3000`
~~~~

## README：服务器部署

~~~~markdown
## 部署到服务器

前提：服务器已安装 Python 3.10+、Node.js 18+ 和 Nginx，
且 Nginx 的站点根目录指向本项目的 out/，监听 80 端口。

### 1. 拉取代码

~~~bash
cd ~/zero-to-tech
git pull
~~~

### 2. 前端：安装依赖、配置、构建

~~~bash
npm install
cp .env.example .env.production
# 修改 .env.production
npm run build
~~~

### 3. 后端：创建环境、安装依赖、配置

~~~bash
cd backend
python3 -m venv --prompt=zero-to-tech .venv
source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env
# 修改 .env
~~~

### 4. 后端后台运行

~~~bash
nohup .venv/bin/fastapi run > backend.log 2>&1 &
~~~

### 5. 放行端口

在云平台安全组或防火墙中放行 8000 端口。

### 6. 验证

访问 `http://服务器IP`，进行一次分析并查看历史；
再用另一个浏览器验证两边的历史互不相见。
~~~~

## README：配置说明

~~~~markdown
## 配置说明

真实配置文件不进 Git，请根据 .env.example 自行创建。

### 前端

开发使用 .env.local，生产构建使用 .env.production。

| 键 | 本地 | 线上 |
| --- | --- | --- |
| NEXT_PUBLIC_API_BASE_URL | http://localhost:8000 | http://服务器IP:8000 |

### 后端

配置文件位于 backend/.env。

| 键 | 本地 | 线上 |
| --- | --- | --- |
| ALLOWED_ORIGINS | http://localhost:3000 | http://服务器IP |
~~~~

写完后，应该假装自己第一次拿到项目，完全按照 README 再走一遍，不依赖记忆补步骤。README 会随着项目继续变化，下一节更换部署方式后还要更新。

## 思考：现在的部署方式好不好？

站点已经上线，但这套最朴素的方案有三个明显问题。

### 第一：后端服务脆弱

`nohup` 只解决 SSH 断开后继续运行，并没有解决：

- 服务器重启后不会自动启动。
- 进程崩溃后不会自动恢复。
- 更新代码后停止和重启都要手动查 PID。
- 查看服务状态不方便。
- 日志位置需要人工记忆。

### 第二：8000 端口直接暴露公网

- 8000 端口没有 Nginx 的访问日志。
- 请求内容没有 HTTPS 加密。
- 没有限流和转发规则。
- 多开放一个端口，就多一份风险。

### 第三：前后端仍然跨源

浏览器 CORS 对跨源请求有很多限制。以后增加域名和 HTTPS 时，跨源问题仍会继续出现。更好的办法是在部署层让前后端同源。

## 下一节要解决什么？

下一节将逐一替换这套朴素方案：

- 用 **systemd** 代替 `nohup`：实现开机自启、崩溃重启、统一查看状态和日志。
- 用 **Nginx 反向代理**把 `/api/` 转发到后端：8000 端口不再对公网开放。
- 前端和后端都通过 80 端口访问，自然变成同源，配置也会更简单。

← 上一节：模块 6.6 状态与会话  
下一节：模块 7.2 常驻与同源——systemd 与 Nginx 反向代理